In [ ]:
import cobra
import sys
sys.path.append('../')
from modelfunctions import *
import os

from memote.support.consistency import check_stoichiometric_consistency, \
    find_inconsistent_min_stoichiometry, \
        find_unconserved_metabolites
from models_emil import * 

wd = os.path.abspath(os.getcwd()).removesuffix('Code/modelUpdates')
figdir = wd + 'Figures/'
models_dir = f'{wd}Data/pcm/'

In [ ]:
def met_overview(pcm, rid):
    print(get_rid(pcm, rid).gene_reaction_rule)
    return pd.DataFrame({m.id:{'Formula': m.formula, 'factor': f, 'Charge': m.charge, 'Name': m.name, 'nRxns': len(m.reactions)} for m,f in get_rid(pcm, rid).metabolites.items()}).T.sort_values('factor')

def check_num_imbalanced(pcm, balance_with_protons = False, return_imbalanced = False):
    c = 0
    #slime_reactions = []
    imbalanced = []
    for r in pcm.reactions:
        # skip import export exchane biomass balance
        if any(r.id.startswith(prefix) for prefix in ['Im_', 'Ex_', 'Exch_', 'DM_', 'Sk_']) or \
                r.id in ['PigmentPool', 'BiomassRxn', 'precursorPool', 'ProteinPool', 'nucleotidePool', 'b6Pool']:
            continue

        # These do not really count since they only supply the biomass reaction
        if r.id.startswith('SLIMEr'):
            #slime_reactions.append(r)
            continue

        try:
            imba = r.check_mass_balance()
        except ValueError as e:
            c += 1
            if return_imbalanced:
                imbalanced.append(r)
            continue

        # 'X' refers to photons, and they cannot be balanced
        # ignore charge imbalances for now
        if not set(imba.keys()).difference(['X']):
            continue

        if imba and not (balance_with_protons and set(imba.keys()) == {'H', 'charge'} and imba['H'] == imba['charge']):
            if return_imbalanced:
                imbalanced.append(r)
            c += 1
    if return_imbalanced:
        return c, imbalanced
    return c

def categorize_gpr(r):
    if not r.gene_reaction_rule:
        return 0
    p, m, a = 0, 0, 0
    for w in r.gene_reaction_rule.split():
        if w.lower() == 'or':
            continue
        if w.startswith('Potri'):
            p += 1
        elif w.startswith('GRMZM'):
            m += 1
        elif w.lower().startswith('at'):
            a += 1
        else:
            return -1
    if a > 0 and p == m == 0:
        return 1
    if m > 0 and a == p == 0:
        return 2
    if p > 0 and a == m == 0:
        return 3
    return -1

def dead_end_rxns(pcm):
    deadM = {m for m in pcm.metabolites if len(m.reactions) == 1}
    rs = {r for m in deadM for r in m.reactions}
    return rs

def find_blocked(pcm, alpha=0.1):
    with pcm:
        for r in pcm.reactions:
            if r.id.startswith('Ex_') or r.id.startswith('Im_') or r.id.startswith('Exch_'):
                r.lower_bound = -1000
                r.upper_bound = 1000
        sol = cobra.flux_analysis.flux_variability_analysis(pcm, fraction_of_optimum=alpha)
        tmp = sol.loc[(sol['minimum'].abs() < 1e-7) & (sol['maximum'].abs() < 1e-7)]
    return tmp

def check_block_problem(pcm, rid):
    r = get_rid(pcm, rid)
    if check_production(pcm, rid, mid_is_rxn=True):
        print('Reactions runs fine')
        return
    for m, f in r.metabolites.items():
        if f < 0:
            if check_production(pcm, rid, mid_is_rxn=True, add_import=[m.id]):
                print('Import of', m.id, 'helped')
        else:
            if check_production(pcm, rid, mid_is_rxn=True, add_export=[m.id]):
                print('Export of', m.id, 'helped')

def create_sub_model(rxn_ls):
    new_model = cobra.Model('Test')

    # make rxn_ls unique
    new_rxn_ls = []
    new_rids = set()
    for r in rxn_ls:
        if r.id not in new_rids:
            new_rxn_ls.append(r)
            new_rids.add(r.id)
    new_model.add_reactions(new_rxn_ls)

    # another copy is required for check_stoichiometric_consistency to update
    return new_model.copy()


In [3]:
pcm1 = cobra.io.read_sbml_model(models_dir + 'pcm.v1.xml')

Set parameter Username
Set parameter LicenseID to value 2852561
Academic license - for non-commercial use only - expires 2027-08-10


In [95]:
pcm = cobra.io.read_sbml_model(models_dir + 'pcm.v2.xml')

In [96]:
print('Biomass production:', pcm.slim_optimize())

numUnb, unb = check_num_imbalanced(pcm, return_imbalanced=True)
print('Number of unbalanced reactions:', numUnb)
if numUnb:
    print(unb)

rs = dead_end_rxns(pcm)
print('Number of reactions with dead-end metabolites:', len(rs))
if rs:
    print(rs)

df = find_blocked(pcm, alpha=0.1)
print('Number of blocked reactions:', df.shape[0])
if not df.empty:
    print(df.index.to_list())

Biomass production: 0.2621094569092053
Number of unbalanced reactions: 0
Number of reactions with dead-end metabolites: 0
Number of blocked reactions: 0


### My way of testing was a bit overkill but helped give me the right idea
I created a much smaller model (18 rxns) that is also inconsistent, but made no sense. However, the memote function find_unconserved_metabolites then gave me the CONST_rubisco metabolite as being unconserved in that model, which gave me the idea that metabolites with 0 mass (the pseudo metabolites acting as constraints, as well as light), might be the problem.

This turned out to be true as can be seen by the model where those metabolites are removed.

In [ ]:
# First do some binary search to find a smaller subset
remaining = list(pcm.reactions.copy())
protected = set()
log_str = ''
with open('log.log', 'w') as file:
    file.write(log_str)

for i in range(12):
    for _ in range(5):
        start, end = 0, len(remaining)
        mid = start + (end - start) // 2
        # Check if the first half is False
        first_half = create_sub_model(set(remaining[:mid]) | protected)
        if not check_stoichiometric_consistency(first_half):
            log_str += f'Found False with first half ({len(first_half.reactions)} rxns)\n'
            remaining = list(first_half.reactions.copy())
        else:
            # If first half evaluates to True, check the second half
            second_half = create_sub_model(set(remaining[mid:]) | protected)
            if not check_stoichiometric_consistency(second_half):
                start = mid + 1
                log_str += f'Found False with second half ({len(second_half.reactions)} rxns)\n'
                remaining = list(second_half.reactions.copy())
            else:
                # if both evaluate to True, fix the first half and continue with the second half
                if i %2 == 0:
                    protected.update(first_half.reactions)
                    remaining = remaining[mid:]
                    log_str += f'Both halves were true, fix first half ({len(first_half.reactions)} rxns)\n'
                else:
                    protected.update(second_half.reactions)
                    remaining = remaining[:mid]
                    log_str += f'Both halves were true, fix second half ({len(second_half.reactions)} rxns)\n'

        remaining = list(set(remaining) - protected)
        with open('log.log', 'a') as file:
            file.write(log_str)
            log_str = ''
            
    remaining = list(set(remaining).union(protected))
    protected = set()

In [ ]:
# Then go through the reactions and try to find groups of 10, then 5, then 1 rxn that can still be excluded
log_str = ''
with open('log2.log', 'w') as file:
    file.write(log_str)
    log_str = ''

for nRxns in [10,5,1]:
    start = 0
    log_str += f'Starting {nRxns} epoch\n*********************\n\n'
    while start + nRxns < len(remaining):
        curr_test_set = remaining[:start] + remaining[(start + nRxns):]
        curr_model = create_sub_model(curr_test_set)
        if not check_stoichiometric_consistency(curr_model):
            log_str += f'Could remove {len(remaining) - len(curr_test_set)} rxns, left with {len(remaining)}\n'
            remaining = curr_test_set
            start = 0
        else:
            start += nRxns
            log_str += f'Could not remove rxns {start} to {(start + nRxns)}\n'

        with open('log2.log', 'a') as file:
            file.write(log_str)
            log_str = ''
min_model = create_sub_model(remaining).copy()
        

In [ ]:
check_stoichiometric_consistency(min_model.copy())

Read LP format model from file /tmp/tmpsigfkelj.lp
Reading time = 0.00 seconds
: 32 rows, 36 columns, 154 nonzeros


No biomass reaction detected. Consistency test results are unreliable if one exists.


False

In [34]:
cobra.flux_analysis.flux_variability_analysis(min_model)

,minimum,maximum
R00725,0.0,0.0
NGAM_h,0.0,0.0
6PGDHNADP_h,0.0,0.0
R02736,0.0,0.0
R00770,0.0,0.0
RBC_h,0.0,0.0
Ru5PK_h,0.0,0.0
PGM_h,0.0,0.0
R01063,0.0,0.0
R01137,0.0,0.0


In [46]:
find_unconserved_metabolites(min_model)

{<Metabolite CONST_rubisco[h] at 0x76b3079c9ee0>}

In [97]:
# Remove constraint-metabolites and light, and write this to disk for testing with MEMOTE
with pcm:
    pcm.remove_metabolites([m for m in pcm.metabolites if m.id.startswith('CONST') or m.id == 'C00205[h]'])
    no_const_no_light = pcm.copy()

print('Full pcm:', check_stoichiometric_consistency(pcm))
print('PCM without pseudo-metabolites and light:', check_stoichiometric_consistency(no_const_no_light))

cobra.io.write_sbml_model(no_const_no_light, models_dir + 'pcm.noConstNoLight.v2.xml')

Read LP format model from file /tmp/tmp5gvc3910.lp
Reading time = 0.00 seconds
: 978 rows, 3272 columns, 13648 nonzeros
Full pcm: False
PCM without pseudo-metabolites and light: True


### Check the same procedure on AraCore and AraGEM:

In [65]:
# Read in aracore
aracore = cobra.io.read_sbml_model(models_dir + '../comparison_models/AraCore_v2_1.wKEGG.xml')

# create a version of aracore with the photon metabolite removed and 
#   check whether either is stoichiometrically consistent
with aracore:
    aracore.remove_metabolites([get_mid(aracore, 'hnu[h]')])
    aracore_no_light = aracore.copy()

print('With light:', check_stoichiometric_consistency(aracore))
print('Without light:', check_stoichiometric_consistency(aracore_no_light))


Read LP format model from file /tmp/tmpmtl_uj1f.lp
Reading time = 0.00 seconds
: 414 rows, 1170 columns, 4396 nonzeros


No biomass reaction detected. Consistency test results are unreliable if one exists.
No biomass reaction detected. Consistency test results are unreliable if one exists.


With light: False
Without light: False


In [63]:
# No; check the metabolites that are inconsistent:
find_inconsistent_min_stoichiometry(aracore)
# Seems to be a different problem in aracore

{(<Metabolite AD[c] at 0x76b325858110>,),
 (<Metabolite Arg[p] at 0x76b30a290170>,),
 (<Metabolite AspP[h] at 0x76b30a868230>,),
 (<Metabolite GCEA[p] at 0x76b2ff0b0050>,),
 (<Metabolite NH4[c] at 0x76b325858380>,),
 (<Metabolite O2[p] at 0x76b2ff0b0110>,),
 (<Metabolite Orn[m] at 0x76b30a868320>,),
 (<Metabolite Ser[c] at 0x76b3258580e0>,),
 (<Metabolite dCDP[c] at 0x76b30a290200>,),
 (<Metabolite dGTP[c] at 0x76b30a2901a0>,)}

In [ ]:
# Also check araGEM
# Read in aracore
aragem = cobra.io.read_sbml_model(models_dir + '../comparison_models/AraGEM_valid.xml')

# create a version of aracore with the photon metabolite removed and 
#   check whether either is stoichiometrically consistent
with aragem:
    aragem.remove_metabolites([get_mid(aragem, 'S_hv_p')])
    aragem_no_light = aragem.copy()

print('With light:', check_stoichiometric_consistency(aragem))
print('Without light:', check_stoichiometric_consistency(aragem_no_light))

# No; check the metabolites that are inconsistent:
find_inconsistent_min_stoichiometry(aragem)
# Seems to be a different problem in AraGEM

Read LP format model from file /tmp/tmp_1elfmxg.lp
Reading time = 0.00 seconds
: 1736 rows, 3202 columns, 12766 nonzeros
With light: False
Without light: False


{(<Metabolite S_2_45_Phospho_45_D_45_glycerate_p at 0x76b313208050>,),
 (<Metabolite S_D_45_erythro_45_3_45_Methylmalate_c at 0x76b316ec8260>,),
 (<Metabolite S_Dextrin_p at 0x76b316ec81a0>,),
 (<Metabolite S_Ethanol_c at 0x76b316ec8110>,),
 (<Metabolite S_Folate_c at 0x76b316ec8170>,
  <Metabolite S_Glycolaldehyde_c at 0x76b30ba546b0>),
 (<Metabolite S_Melibiitol_c at 0x76b310e600b0>,),
 (<Metabolite S_alpha_45_D_45_Glucose_c at 0x76b3099a8200>,),
 (<Metabolite S_cis_45_3_44_4_45_leucopelargonidin_58_NADP_43__32_4_45_oxidoreductase_59__32_Luteoforol_c at 0x76b3099a8260>,),
 (<Metabolite S_dCTP_c at 0x76b3099a8080>,),
 (<Metabolite S_trans_45_Dodec_45_2_45_enoyl_45__91_acp_93__p at 0x76b30a9a8200>,)}